::: {.sb-lesson-links}
[Exercises →](../exercises/titanic.ipynb)
[Quiz →](../quiz/quiz.qmd)
[All files of this section (zip)](../../files/06-ensemble-methods.zip)
:::

::: {.sb-downloads}
[Download this notebook](../downloads/lesson-voting-and-stacking.ipynb){download="lesson-voting-and-stacking.ipynb"}  
Data: [titanic.csv](../downloads/data/titanic.csv){download="titanic.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

<!-- TOC START min:2 max:4 link:true asterisk:true update:true -->

* [What you will learn in this course](#what-you-will-learn-in-this-course)
  * [Voting / Averaging](#voting--averaging)
  * [Stacking](#stacking)
<!-- TOC END -->

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.datasets import make_friedman1
from sklearn.ensemble import StackingClassifier, VotingClassifier, VotingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import cross_val_predict, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"

## What you will learn in this course ##

This course introduces two very useful techniques in data science that make your models more stable and improve their performance: averaging/voting and stacking. These techniques are very simple to implement in practice, and require very few new commands.

## Voting / Averaging

We speak of a **Voting classifier** for a classification problem, when the target variable is qualitative. The idea is simple: instead of training a single model to predict the target variable, we train several different models (for example, a Naive Bayes, an SVM and a Random Forest). Each model gives a prediction $\hat{y}_1,\hat{y}_2,\hat{y}_3$ and has a score $S_1,S_2,S_3$. The voting classifier is an aggregated model whose predictions come from a weighted vote of the different models: each model's vote is weighted by its score (the better a model performs, the more weight it has; the worse it performs, the less weight it has). 



::: {.callout-tip title="Intuition"}
This is the "wisdom of the crowd". If you ask three doctors who were trained differently, they will not all make the same mistakes: when one is wrong, the two others often catch it. A majority vote is then right more often than each doctor alone. The key word is *different*: three copies of the same model would always agree, and their vote would add nothing.
:::



Mathematically, we can write: $\hat{y}=argmax_{m\in M}(S_1\times(\hat{y}_1=m)+S_2\times(\hat{y}_2=m)+S_3\times(\hat{y}_3=m))$
 where $M$ is the set of modalities taken by $y$.

**See it with data 🔬 Three different models on the Titanic**

Let's predict who survived the Titanic (`titanic.csv`, 891 passengers) from 6 simple features, with three models that see the data in very different ways: a logistic regression (a straight border), a k-nearest neighbors (looks at similar passengers) and a decision tree (a sequence of yes/no questions). 30% of the passengers are kept aside as a test set.

In [ ]:
df = pd.read_csv("../data/titanic.csv")
df["Sex"] = (df["Sex"] == "female").astype(int)             # 1 = female, 0 = male
df["Age"] = df["Age"].fillna(df["Age"].median())
X = df[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]]
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

models = {"logistic regression": make_pipeline(StandardScaler(), LogisticRegression()),
          "k-NN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
          "decision tree": DecisionTreeClassifier(max_depth=5, random_state=0)}
preds = pd.DataFrame({name: m.fit(X_train, y_train).predict(X_test) for name, m in models.items()})
single_scores = {name: (preds[name].values == y_test.values).mean() for name in models}
pd.Series(single_scores, name="test accuracy").round(3)

logistic regression    0.813
k-NN                   0.810
decision tree          0.825
Name: test accuracy, dtype: float64

In [ ]:
split = (preds.nunique(axis=1) > 1).values            # passengers on which the 3 models do not agree
majority = (preds.sum(axis=1) >= 2).astype(int).values  # majority vote of the 3 models
print(f"The three models disagree on {split.sum()} of the {len(preds)} test passengers ({split.mean():.1%}).")
print("Accuracy on these passengers only:")
for name in models:
    print(f"  {name:<20} {(preds[name].values[split] == y_test.values[split]).mean():.3f}")
print(f"  {'majority vote':<20} {(majority[split] == y_test.values[split]).mean():.3f}")

The three models disagree on 38 of the 268 test passengers (14.2%).
Accuracy on these passengers only:
  logistic regression  0.579
  k-NN                 0.553
  decision tree        0.658
  majority vote        0.789


The three models have close scores (0.813, 0.810 and 0.825), but they do not make the same mistakes: they disagree on 38 of the 268 test passengers (14.2%). On those difficult passengers, each model alone is right 55% to 66% of the time, while the majority vote is right 79% of the time. That is exactly what a vote needs: models that are good, *and* different.

**See it with data 🔬 Hard vs soft voting with `VotingClassifier`**

scikit-learn offers two ways to vote. **Hard voting** counts the predicted classes (the majority wins), optionally with a weight per model, like the scores $S_1, S_2, S_3$ of the formula above. **Soft voting** averages the predicted *probabilities* of the models, so a very confident model weighs more than a hesitant one. Here the scores $S_m$ are cross-validated accuracies, computed on the training set only.

In [ ]:
estimators = list(models.items())
scores = [cross_val_score(m, X_train, y_train, cv=5).mean() for m in models.values()]   # S1, S2, S3
print("Scores S1, S2, S3:", np.round(scores, 3))

voters = {"hard voting": VotingClassifier(estimators, voting="hard"),
          "hard voting weighted by S": VotingClassifier(estimators, voting="hard", weights=scores),
          "soft voting": VotingClassifier(estimators, voting="soft")}
vote_scores = {name: v.fit(X_train, y_train).score(X_test, y_test) for name, v in voters.items()}
pd.Series(vote_scores, name="test accuracy").round(3)

Scores S1, S2, S3: [0.783 0.807 0.811]


hard voting                  0.843
hard voting weighted by S    0.843
soft voting                  0.832
Name: test accuracy, dtype: float64

Hard voting reaches 0.843 test accuracy: better than each of the three models alone (best: 0.825). Weighting by the scores changes nothing here (0.843 again): with three models of similar scores (0.783 to 0.811), two votes always outweigh one, so the weighted vote is still a majority vote. Soft voting gets 0.832 on this test set. Keep in mind that the test set has only 268 passengers, so one passenger is worth 0.004 of accuracy: differences of 0.01 are a handful of passengers.

Averaging is used for regression, when the target variable is quantitative. Here too, we train several different models to get various predictions for the target variable $y$ (for example, a linear regression, an SVM and a Random Forest). Each of these models gives a prediction for $y$: $\hat{y}_1,\hat{y}_2,\hat{y}_3$ and has a score: $S_1,S_2,S_3$. The final prediction is the average of the models' predictions, weighted by their scores. Mathematically, this can be written as follows: $\hat{y}=\frac{1}{S_1+S_2+S_3}\times(S_1\times\hat{y}_1+S_2\times\hat{y}_2+S_3\times\hat{y}_3)$. 

You can create a voting classifier in Python with the class described here: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html.

For the voting regressor, follow this link: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingRegressor.html#sklearn.ensemble.VotingRegressor

**See it with data 🔬 Averaging three regressors with `VotingRegressor`**

The same idea for a quantitative target, on a simulated dataset (`make_friedman1`: 1,000 observations, 10 features). `VotingRegressor` simply averages the predictions of a linear regression, a k-NN and a decision tree.

In [ ]:
X_r, y_r = make_friedman1(n_samples=1000, noise=1.0, random_state=0)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_r, y_r, test_size=0.3, random_state=0)

regressors = {"linear regression": LinearRegression(),
              "k-NN": KNeighborsRegressor(n_neighbors=10),
              "decision tree": DecisionTreeRegressor(max_depth=6, random_state=0)}
regressors["average of the three"] = VotingRegressor(list(regressors.items()))
r2 = {name: m.fit(Xr_train, yr_train).score(Xr_test, yr_test) for name, m in regressors.items()}
pd.Series(r2, name="test R²").round(3)

linear regression       0.743
k-NN                    0.696
decision tree           0.658
average of the three    0.781
Name: test R², dtype: float64

Alone, the three models reach a test $R^2$ of 0.743 (linear regression), 0.696 (k-NN) and 0.658 (decision tree). Their simple average reaches 0.781: better than the best of them. Each model makes errors in its own way (the linear model is too rigid, the tree predicts in steps, k-NN is blurry), and averaging cancels part of these errors.

### Stacking

![Voting combines the predictions with a fixed rule; stacking learns how to combine them](../img/voting_stacking.svg)




::: {.callout-tip title="Intuition"}
Voting combines the models with a fixed rule (a majority, an average). Stacking hires a **manager**: a second-level model that looks at what each first-level model predicted, compares it with the truth, and *learns* how much to trust each one. To judge fairly, the manager must see the models' predictions on data they were not trained on, otherwise a model that learned the training set by heart would look perfect.
:::



The idea of stacking is to train several layers of models to predict our target variable. This technique works for regression problems as well as classification problems. The principle is simple. First, we split the training sample into two parts, which we call *part 1* and *part 2*. Then we choose several different models and train them on *part 1*. Once these models are trained, we build a new training dataset from the predictions that each of them makes on the data of *part 2*. Splitting the training set in two ensures that the first-level and second-level models are trained on different observations, which limits overfitting. In this new dataset, the explanatory variables of each *part 2* observation are replaced by the predictions of the first-level models. We then choose a new model and train it on this new dataset (built from *part 2*) to predict the target variable. This model can perform better, since the variables it is trained on are themselves predictions of $y$, and contain information computed specifically to describe $y$.

This method describes a two-level Stacking, i.e. a first set of models (first order) and a final model (second order), also called a metamodel. The principle can be extended to more stacking layers: you just need to split the training set into as many parts as necessary, so that each new layer is trained on data used only for this layer.

**See it with data 🔬 Stacking with scikit-learn's `StackingClassifier`**

scikit-learn has stacking built in (the `vecstack` way is shown just below). `StackingClassifier` trains the first-level models, and gives the meta-model their *out-of-fold* predictions: with `cv=5`, the training set is cut into 5 parts, and each passenger is predicted by models trained on the 4 other parts. This plays the role of the *part 1* / *part 2* split, without throwing away any data. Our meta-model is a logistic regression, and its coefficients tell us how much it trusts each first-level model.

In [ ]:
stack = StackingClassifier(estimators, final_estimator=LogisticRegression(), cv=5)
stack.fit(X_train, y_train)
print(f"Stacking test accuracy = {stack.score(X_test, y_test):.3f}")

trust = pd.Series(stack.final_estimator_.coef_[0], index=list(models), name="meta-model coefficient")
trust.round(2)

Stacking test accuracy = 0.832


logistic regression    1.99
k-NN                   1.48
decision tree          1.86
Name: meta-model coefficient, dtype: float64

In [ ]:
summary = {**single_scores, **vote_scores, "stacking": stack.score(X_test, y_test)}
fig = go.Figure(go.Bar(x=list(summary.values()), y=list(summary), orientation="h",
                       text=[f"{v:.3f}" for v in summary.values()], textposition="outside"))
fig.update_layout(title="Titanic: test accuracy of single models and ensembles", xaxis_title="test accuracy",
                  yaxis_title="model", xaxis_range=[0.75, 0.87], yaxis_autorange="reversed")
fig.show()

The meta-model gives positive and fairly balanced coefficients to the three models: 1.99 for the logistic regression, 1.86 for the decision tree and 1.48 for k-NN, the one it trusts least. Stacking reaches 0.832 test accuracy: better than each single model, the same as soft voting, and a little below hard voting (0.843) on this test set. Stacking is not guaranteed to win: on a small dataset with similar models, a simple vote is a very strong competitor, and it is cheaper to train.

**See it with data 🔬 Why the meta-model must not see the training predictions**

What if we skipped the split, and trained the meta-model on the predictions that the first-level models make *on their own training data*? Let's compare the coefficients of the meta-model in both cases.

In [ ]:
in_sample = np.column_stack([m.fit(X_train, y_train).predict_proba(X_train)[:, 1] for m in models.values()])
out_of_fold = np.column_stack([cross_val_predict(m, X_train, y_train, cv=5, method="predict_proba")[:, 1]
                               for m in models.values()])
coefs = pd.DataFrame({"trained on in-sample predictions": LogisticRegression().fit(in_sample, y_train).coef_[0],
                      "trained on out-of-fold predictions": LogisticRegression().fit(out_of_fold, y_train).coef_[0]},
                     index=list(models))
coefs.round(2)

,trained on in-sample predictions,trained on out-of-fold predictions
logistic regression,0.32,1.99
k-NN,1.89,1.48
decision tree,3.89,1.86


Trained on in-sample predictions, the meta-model puts a huge coefficient on the decision tree (3.89) and almost ignores the logistic regression (0.32): on its own training data the tree looks excellent, because it has partly learned those passengers by heart. That trust is not deserved on new passengers. With out-of-fold predictions, the coefficients are balanced (1.99, 1.48, 1.86), and they are exactly the ones `StackingClassifier` learned above. This is why stacking always trains the meta-model on predictions made on data the first-level models have not seen.

To build a Stacking model, you can use the `stacking` function from the `vecstack` library ("from vecstack import stacking"), as follows:

```python
stacking(models,
X_train, y_train, X_test,
regression=False, 
mode='oof_pred_bag', 
needs_proba=False,
save_dir=None, 
metric=accuracy_score, 
n_folds=4, 
stratified=True,
shuffle=True,  
random_state=0,    
verbose=2)
```

The stacking function takes the following arguments:

**models**: list of the model instances of the first layer

**X_train**, **y_train**, **X_test**: test and train data

**regression**: boolean indicating whether it is a regression or a classification problem

**mode**: how the first-layer predictions are computed. Here, it's the 'out of fold' technique: the training set is split into folds, and each observation is predicted by a model trained on the other folds, as in cross-validation (with 'oof_pred_bag', the test-set predictions are averaged over the folds).

**needs_proba**: boolean indicating whether to output probabilities or not

**save_dir**: folder in which to save the results once computed (None means no saving)

**metric**: metric used to evaluate the first-layer models

**n_folds**: number of folds for cross-validation

**stratified**: boolean indicating whether to stratify the folds according to the modalities of the target variable or not

**shuffle**: boolean indicating whether to shuffle the observations or not

**random_state**: the seed of the pseudo-random number generator

**verbose**: int indicating the level of information returned when the command is executed, from 0 (no info) to 2 (maximum info).

::: {.sb-next}
**[Next: Summary: boosting →](../lessons/summary-of-boosting.ipynb)**
:::